In [0]:
%run ../00-common/01.environment-config

In [0]:
target_table= f"{catalog_name}.{gold_schema}.fact_session_results"

In [0]:
from pyspark.sql import functions as F

In [0]:
results_df =( spark.table(f"{catalog_name}.{silver_schema}.results")
                .withColumn("session_type", F.lit("RACE"))
                .drop("race_name", "race_date", "ingestion_timestamp", "source_file")
)

In [0]:
sprints_df =( spark.table(f"{catalog_name}.{silver_schema}.sprints")
                .withColumn("session_type", F.lit("SPRINT"))
                .drop("race_name", "race_date", "ingestion_timestamp", "source_file")
)

In [0]:
results_sprints_df= results_df.unionByName(sprints_df)

In [0]:
fact_session_results_df=(
    results_sprints_df
        .withColumn("is_win", F.col("final_position")==1)
        .withColumn("is_podium", F.col("final_position").between(1,3))
        .withColumn("has_points", F.col("points")>0)
)

In [0]:
display(fact_session_results_df.filter("season==2025"))

In [0]:
%sql
drop table formula1.gold.fact_session_results

In [0]:
 (
     fact_session_results_df
     .write
     .format("delta")
     .mode("overwrite")
     .saveAsTable(target_table)
 )

In [0]:
display(spark.table(target_table))